# Assignment 2: Data Parsing, Cleansing and Integration
## Task 3 — Schema Alignment and Data Integration
#### Student Name: Ngo Trong Nhan
#### Student ID: s4196976

Date: 2026-04-14

Version: 1.0

Environment: Python 3 and Jupyter notebook

Libraries used:
* pandas
* numpy
* copy

## Introduction

This notebook integrates two job advertisement datasets:
- **df_task2** (`s4196976_data_4_solution.csv`) — 50 750 cleaned records produced in Tasks 1 & 2.
- **df_task3** (`data_4.csv`) — 5 000 raw records from `www.jobhuntlisting.com`.

Key schema conflicts resolved:
1. **Column naming** — df_task3 uses different names for every field.
2. **Salary unit** — df_task3 stores a *monthly* figure; global schema requires *annual* salary.
3. **ContractType derivation** — derived from `Full-Time Equivalent (FTE)` (1.0 → `full_time`, < 1.0 → `part_time`).
4. **ContractTime normalisation** — raw values mapped to canonical set.
5. **Category mapping** — abbreviated labels mapped to global canonical strings.
6. **Missing Id** — 8-digit IDs generated without collision against df_task2.
7. **Missing SourceName** — set to `www.jobhuntlisting.com`.

## Importing libraries

In [1]:
import copy
import numpy as np
import pandas as pd

# Helper
Helper classes used throughout Task 3.

In [2]:
class SemanticLayer:
    """
    Pure mapping class — no mutable state.

    Encapsulates every rule needed to translate df_task3's raw values into the
    global schema defined by df_task2 / Table 1 of the assignment spec.

    Design note
    -----------
    Category keys are df_task3's abbreviated sector labels (the non-canonical
    source). Target values are the exact global strings from Table 1.
    Keeping the mapping here (not scattered across SchemaResolver) makes
    it easy to audit and extend.
    """

    # Column rename map  (df_task3 raw name -> global schema name)
    COLUMN_RENAME: dict = {
        "Job Title"                  : "Title",
        "Monthly Payment"            : "Salary",
        "Opening"                    : "OpenDate",
        "Closing"                    : "CloseDate",
        "Organisation"               : "Company",
        "Type"                       : "ContractTime",
        "Full-Time Equivalent (FTE)" : "ContractType",
    }

    # Category  (df_task3 abbreviated label -> global canonical string)
    __CATEGORY_MAP: dict = {
        "Health"                : "Healthcare & Nursing Jobs",
        "Engineering"           : "Engineering Jobs",
        "Hospitality"           : "Hospitality & Catering Jobs",
        "Information Technology": "IT Jobs",
        "Finance"               : "Accounting & Finance Jobs",
        "Marketing"             : "PR, Advertising & Marketing Jobs",
        "Sales"                 : "Sales Jobs",
        "Education"             : "Teaching Jobs",
    }

    # Reverse category map  (df_task2 canonical string -> df_task3 abbreviated label)
    __CATEGORY_REVERSE_MAP: dict = {
        "Healthcare & Nursing Jobs"        : "Health",
        "Engineering Jobs"                 : "Engineering",
        "Hospitality & Catering Jobs"      : "Hospitality",
        "IT Jobs"                          : "Information Technology",
        "Accounting & Finance Jobs"        : "Finance",
        "PR, Advertising & Marketing Jobs" : "Marketing",
        "Sales Jobs"                       : "Sales",
        "Teaching Jobs"                    : "Education",
    }

    # ContractTime  (df_task3 'Type' raw value -> global canonical string)
    __CONTRACT_TIME_MAP: dict = {
        "permanent"           : "permanent",
        "fixed term contract" : "contract",
        "contract"            : "contract",
    }

    @classmethod
    def map_category(cls, value: str) -> str:
        """Map a df_task3 abbreviated category label to the global canonical string.

        Args:
            value: Raw category string from df_task3 (e.g. 'Health').

        Returns:
            Canonical category string (e.g. 'Healthcare & Nursing Jobs'),
            or 'non-specified' if not in the map.
        """
        return cls.__CATEGORY_MAP.get(value, "non-specified")

    @classmethod
    def map_category_to_general(cls, value: str) -> str:
        """Map a df_task2 canonical category string to the general df_task3 label.

        Args:
            value: Canonical category string from df_task2 (e.g. 'Healthcare & Nursing Jobs').

        Returns:
            General category label (e.g. 'Health'), or the original value if not in the map.
        """
        return cls.__CATEGORY_REVERSE_MAP.get(value, value)

    @classmethod
    def map_contract_time(cls, type_series: pd.Series) -> pd.Series:
        """Normalise df_task3 Type values to the global ContractTime domain.

        Raw values observed: 'Permanent', 'Fixed Term Contract', NaN.
        Strategy: lower-strip then look up; unrecognised (incl. NaN) -> 'non-specified'.

        Args:
            type_series: The raw Type column as a pd.Series.

        Returns:
            pd.Series with values from {'permanent', 'contract', 'non-specified'}.
        """
        normalised = type_series.astype(str).str.strip().str.lower()
        return normalised.map(lambda v: cls.__CONTRACT_TIME_MAP.get(v, "non-specified"))

    @classmethod
    def convert_fte_to_contract_type(cls, fte_series: pd.Series) -> pd.Series:
        """Derive ContractType from the Full-Time Equivalent metric.

        FTE semantics (standard HR definition):
            1.0  -> full working week (~37-40 h) -> full_time
            <1.0 -> reduced hours (0.2 to 0.8)  -> part_time
            NaN  -> no information               -> non-specified

        Args:
            fte_series: The Full-Time Equivalent (FTE) column as a pd.Series.

        Returns:
            pd.Series with values from {'full_time', 'part_time', 'non-specified'}.
        """
        fte = pd.to_numeric(fte_series, errors="coerce")
        return fte.map(
            lambda x: "non-specified" if pd.isna(x)
            else "full_time" if x >= 1.0
            else "part_time"
        )

    @classmethod
    def monthly_to_annual_salary(cls, salary_series: pd.Series) -> pd.Series:
        """Convert a monthly salary to an annual figure (multiply by 12).

        The global schema stores annual salary rounded to 2 decimal places.

        Args:
            salary_series: The Monthly Payment column as a pd.Series.

        Returns:
            pd.Series of annual salary values (float64, 2 dp).
        """
        return (pd.to_numeric(salary_series, errors="coerce") * 12).round(2)

In [3]:
class SchemaResolver:
    """
    Owns a working copy of df_task3 and exposes methods that resolve each
    schema conflict in place, logging every change to self.schema_log.

    Mirrors the JobAuditor pattern from Task 1/2:
    - self.df is the live DataFrame (mutated by each resolve_* method).
    - self.schema_log accumulates one record per resolved conflict.
    - copy() produces an independent clone for safe experimentation.
    - resolve_all() is the full pipeline.
    """

    # Global schema column order (Table 1 of the spec)
    __GLOBAL_COLUMNS: list = [
        "Id", "Title", "Location", "Company",
        "ContractType", "ContractTime", "Category",
        "Salary", "OpenDate", "CloseDate", "SourceName",
    ]

    __SOURCE_NAME: str = "www.jobhuntlisting.com"

    def __init__(self, df_task2: pd.DataFrame, df_task3: pd.DataFrame) -> None:
        """Attach working copies of both source DataFrames and initialise an empty schema log.

        Args:
            df_task2: Cleaned output from Tasks 1 & 2 (used by resolve_categories).
            df_task3: Raw df_task3 as loaded from data/data_4.csv.

        Returns:
            None (sets self.df_task2, self.df, and self.schema_log).
        """
        self.df_task2: pd.DataFrame = df_task2.copy()
        self.df: pd.DataFrame = df_task3.copy()
        self.schema_log: list = []

    def copy(self) -> "SchemaResolver":
        """Clone this resolver so downstream changes do not affect the original.

        Args:
            None.

        Returns:
            New SchemaResolver with deep-copied df and schema_log.
        """
        other = object.__new__(self.__class__)
        other.df_task2 = self.df_task2.copy(deep=True)
        other.df = self.df.copy(deep=True)
        other.schema_log = copy.deepcopy(self.schema_log)
        return other

    def _log(self, conflict: str, action: str, detail: str = "") -> None:
        """Append one entry to self.schema_log.

        Args:
            conflict: Short label for the conflict type (e.g. 'naming').
            action:   Description of what was done.
            detail:   Optional extra information.

        Returns:
            None.
        """
        self.schema_log.append({"conflict": conflict, "action": action, "detail": detail})

    def get_schema_log_df(self) -> pd.DataFrame:
        """Return the accumulated schema-conflict log as a DataFrame.

        Args:
            None.

        Returns:
            pd.DataFrame with columns conflict, action, detail.
        """
        return pd.DataFrame(self.schema_log, columns=["conflict", "action", "detail"])

    def resolve_column_names(self, verbose: bool = False) -> None:
        """Rename df_task3 columns to match the global schema (Conflict 1).

        Uses SemanticLayer.COLUMN_RENAME as the authoritative rename map.

        Args:
            verbose: If True, print the rename mapping applied.

        Returns:
            None (renames columns in self.df in place).
        """
        self.df.rename(columns=SemanticLayer.COLUMN_RENAME, inplace=True)
        self._log(
            conflict="naming",
            action="renamed columns",
            detail=str(SemanticLayer.COLUMN_RENAME),
        )
        if verbose:
            print("[resolve_column_names] Applied:", SemanticLayer.COLUMN_RENAME)

    def resolve_categories(self, verbose: bool = False) -> None:
        """Normalise Category to df_task3's general labels for both datasets (Conflict 2).

        df_task3 already carries the general labels (e.g. 'Health'), so no mapping is
        needed for self.df.  df_task2 uses verbose canonical strings
        (e.g. 'Healthcare & Nursing Jobs') which are converted here to the same general
        labels via SemanticLayer.map_category_to_general.

        Args:
            verbose: If True, print before/after unique value counts for both DataFrames.

        Returns:
            None (updates self.df_task2['Category'] in place; self.df['Category'] unchanged).
        """
        # df_task3 – already in the target general format, no mapping required
        task3_cats = self.df["Category"].unique().tolist()

        # df_task2 – convert verbose canonical strings -> general labels
        before_task2 = self.df_task2["Category"].unique().tolist()
        self.df_task2["Category"] = self.df_task2["Category"].map(
            SemanticLayer.map_category_to_general
        )
        after_task2 = self.df_task2["Category"].unique().tolist()

        self._log(
            conflict="category values",
            action="converted df_task2 canonical labels to general labels (df_task3 format)",
            detail=f"df_task2 before={before_task2} -> after={after_task2}; df_task3 unchanged={task3_cats}",
        )
        if verbose:
            print(f"[resolve_categories] df_task3 categories (unchanged): {task3_cats}")
            print(f"[resolve_categories] df_task2 before: {before_task2}")
            print(f"[resolve_categories] df_task2 after : {after_task2}")

    def resolve_salary(self, verbose: bool = False) -> None:
        """Convert monthly salary to annual by multiplying by 12 (Conflict 3).

        Args:
            verbose: If True, print the salary range before and after conversion.

        Returns:
            None (updates self.df['Salary'] in place).
        """
        before_min = self.df["Salary"].min()
        before_max = self.df["Salary"].max()
        self.df["Salary"] = SemanticLayer.monthly_to_annual_salary(self.df["Salary"])
        after_min = self.df["Salary"].min()
        after_max = self.df["Salary"].max()
        self._log(
            conflict="salary unit",
            action="converted monthly -> annual (x12, rounded 2 dp)",
            detail=f"range [{before_min}, {before_max}] -> [{after_min}, {after_max}]",
        )
        if verbose:
            print(f"[resolve_salary] monthly range: [{before_min:.2f}, {before_max:.2f}]")
            print(f"[resolve_salary] annual  range: [{after_min:.2f}, {after_max:.2f}]")

    def resolve_contract_type(self, verbose: bool = False) -> None:
        """Derive ContractType from the Full-Time Equivalent column (Conflict 4).

        After resolve_column_names the FTE column has already been renamed to
        ContractType; this method replaces its numeric values with canonical strings.

        Args:
            verbose: If True, print the resulting value distribution.

        Returns:
            None (updates self.df['ContractType'] in place).
        """
        self.df["ContractType"] = SemanticLayer.convert_fte_to_contract_type(
            self.df["ContractType"]
        )
        dist = self.df["ContractType"].value_counts().to_dict()
        self._log(
            conflict="ContractType (FTE derivation)",
            action="FTE 1.0->full_time, <1.0->part_time, NaN->non-specified",
            detail=str(dist),
        )
        if verbose:
            print(f"[resolve_contract_type] distribution: {dist}")

    def resolve_contract_time(self, verbose: bool = False) -> None:
        """Normalise raw ContractTime (Type) values to the global canonical set (Conflict 5).

        After resolve_column_names the Type column is already named ContractTime;
        this method normalises its string values.

        Args:
            verbose: If True, print before/after value distributions.

        Returns:
            None (updates self.df['ContractTime'] in place).
        """
        before = self.df["ContractTime"].value_counts(dropna=False).to_dict()
        self.df["ContractTime"] = SemanticLayer.map_contract_time(self.df["ContractTime"])
        after = self.df["ContractTime"].value_counts().to_dict()
        self._log(
            conflict="ContractTime values",
            action="normalised to {permanent, contract, non-specified}",
            detail=f"before={before} -> after={after}",
        )
        if verbose:
            print(f"[resolve_contract_time] before: {before}")
            print(f"[resolve_contract_time] after : {after}")

    def generate_ids(self, existing_ids: pd.Series, verbose: bool = False) -> None:
        """Generate unique 8-digit IDs for df_task3 records (Conflict 6).

        New IDs are assigned sequentially starting from max(existing_ids) + 1,
        guaranteeing no collision with df_task2.

        Args:
            existing_ids: The Id column of df_task2 (used to find the next available ID).
            verbose:      If True, print the generated ID range.

        Returns:
            None (adds / overwrites self.df['Id'] in place).
        """
        start = int(existing_ids.max()) + 1
        new_ids = np.arange(start, start + len(self.df), dtype=np.int64)
        self.df["Id"] = new_ids
        self._log(
            conflict="missing Id",
            action=f"generated sequential IDs from {new_ids[0]} to {new_ids[-1]}",
            detail=f"collision check: {int(pd.Series(new_ids).isin(existing_ids).sum())} overlaps",
        )
        if verbose:
            print(f"[generate_ids] new Id range: [{new_ids[0]}, {new_ids[-1]}]")
            print(f"[generate_ids] all 8-digit: "
                  f"{all(10_000_000 <= i <= 99_999_999 for i in [new_ids[0], new_ids[-1]])}")

    def add_source_name(self, verbose: bool = False) -> None:
        """Add the SourceName column pointing to www.jobhuntlisting.com (Conflict 7).

        Args:
            verbose: If True, print the value assigned.

        Returns:
            None (adds self.df['SourceName'] in place).
        """
        self.df["SourceName"] = self.__SOURCE_NAME
        self._log(
            conflict="missing SourceName",
            action=f"set SourceName = '{self.__SOURCE_NAME}' for all rows",
        )
        if verbose:
            print(f"[add_source_name] SourceName = '{self.__SOURCE_NAME}'")

    def fill_missing_company(self, verbose: bool = False) -> None:
        """Replace NaN in Company with the placeholder 'non-specified'.

        Args:
            verbose: If True, print the number of rows filled.

        Returns:
            None (updates self.df['Company'] in place).
        """
        n_missing = self.df["Company"].isna().sum()
        self.df["Company"] = self.df["Company"].fillna("non-specified")
        self._log(
            conflict="missing Company",
            action=f"filled {n_missing} NaN values with 'non-specified'",
        )
        if verbose:
            print(f"[fill_missing_company] filled {n_missing} NaN values")

    def parse_dates(self, verbose: bool = False) -> None:
        """Parse OpenDate and CloseDate columns to datetime64.

        Args:
            verbose: If True, print the dtype of each date column after parsing.

        Returns:
            None (updates date columns in self.df in place).
        """
        for col in ("OpenDate", "CloseDate"):
            self.df[col] = pd.to_datetime(self.df[col], errors="coerce")
        self._log(conflict="date dtype", action="parsed OpenDate and CloseDate to datetime64")
        if verbose:
            print(f"[parse_dates] OpenDate  dtype: {self.df['OpenDate'].dtype}")
            print(f"[parse_dates] CloseDate dtype: {self.df['CloseDate'].dtype}")

    def reorder_columns(self, verbose: bool = False) -> None:
        """Reorder self.df columns to match the global schema order.

        Args:
            verbose: If True, print the resulting column list.

        Returns:
            None (reorders self.df columns in place).
        """
        self.df = self.df[self.__GLOBAL_COLUMNS]
        self._log(conflict="column order", action="reordered to global schema",
                  detail=str(self.__GLOBAL_COLUMNS))
        if verbose:
            print(f"[reorder_columns] final columns: {list(self.df.columns)}")

    # ------------------------------------------------------------------
    # Full pipeline
    # ------------------------------------------------------------------

    def resolve_all(self, existing_ids: pd.Series, verbose: bool = False) -> None:
        """Run the complete schema-resolution pipeline in the correct order.

        Step order matters: column rename must come before any method that
        references the new column names.

        Args:
            existing_ids: Id series from df_task2, passed to generate_ids.
            verbose:      Forwarded to every individual resolver.

        Returns:
            None.
        """
        self.resolve_column_names(verbose=verbose)    # must be first
        self.resolve_categories(verbose=verbose)
        self.resolve_salary(verbose=verbose)
        self.resolve_contract_type(verbose=verbose)
        self.resolve_contract_time(verbose=verbose)
        self.generate_ids(existing_ids, verbose=verbose)
        self.add_source_name(verbose=verbose)
        self.fill_missing_company(verbose=verbose)
        self.parse_dates(verbose=verbose)
        self.reorder_columns(verbose=verbose)

In [4]:
class DataIntegrator:
    """
    Owns the unified DataFrame produced by merging df_task2 and the resolved
    df_task3, and exposes methods to detect and remove data-level conflicts.

    - self.df is the live unified DataFrame.
    - self.dedup_log accumulates one record per deduplication step.
    - copy() produces an independent clone for safe experimentation.
    """

    def __init__(self, df_task2: pd.DataFrame, df_task3_resolved: pd.DataFrame) -> None:
        """Concatenate df_task2 and the resolved df_task3 into the unified table.

        Args:
            df_task2:          Cleaned output from Tasks 1 & 2.
            df_task3_resolved: df_task3 after full schema resolution.

        Returns:
            None (sets self.df and self.dedup_log).
        """
        self.df: pd.DataFrame = pd.concat(
            [df_task2, df_task3_resolved], ignore_index=True
        )
        self.dedup_log: list = []

    def copy(self) -> "DataIntegrator":
        """Clone this integrator so downstream changes do not affect the original.

        Args:
            None.

        Returns:
            New DataIntegrator with deep-copied df and dedup_log.
        """
        other = object.__new__(self.__class__)
        other.df = self.df.copy(deep=True)
        other.dedup_log = copy.deepcopy(self.dedup_log)
        return other

    def get_dedup_log_df(self) -> pd.DataFrame:
        """Return the deduplication log as a DataFrame.

        Args:
            None.

        Returns:
            pd.DataFrame with columns step, rows_before, rows_removed, rows_after, detail.
        """
        return pd.DataFrame(
            self.dedup_log,
            columns=["step", "rows_before", "rows_removed", "rows_after", "detail"],
        )

    def remove_complete_duplicates(self, verbose: bool = False) -> None:
        """Remove rows where every column value is identical to another row.

        Strategy: keep the first occurrence (keep='first').

        Args:
            verbose: If True, print duplicate count and sample rows.

        Returns:
            None (removes duplicate rows from self.df and resets index).
        """
        before = len(self.df)
        mask = self.df.duplicated(keep=False)
        if verbose and mask.sum() > 0:
            print(f"[remove_complete_duplicates] {mask.sum()} duplicate rows found:")
        
        self.df = self.df.drop_duplicates(keep="first").reset_index(drop=True)
        removed = before - len(self.df)
        self.dedup_log.append({
            "step"        : "complete duplicates",
            "rows_before" : before,
            "rows_removed": removed,
            "rows_after"  : len(self.df),
            "detail"      : "all-column exact match; kept first occurrence",
        })
        if verbose:
            print(f"[remove_complete_duplicates] removed {removed} rows -> {len(self.df)} remaining")

    def remove_near_duplicates(
        self,
        keys: list = None,
        verbose: bool = False,
    ) -> None:
        """Remove near-duplicate rows that share the same business-key values.

        A near-duplicate is defined as two rows with identical values in every
        field of keys — the same job ad appearing in both sources with a
        different Id assigned by each system.

        Strategy: sort by Id ascending so the df_task2 record (lower ID,
        from the original cleaned data) is always kept.

        Args:
            keys:    Columns that define a near-duplicate.
                     Defaults to ['Title', 'Company', 'Location', 'OpenDate'].
            verbose: If True, print the count and a sample of near-duplicate pairs.

        Returns:
            None (removes duplicate rows from self.df and resets index).
        """
        if keys is None:
            keys = ["Title", "Company", "Location", "OpenDate"]

        before = len(self.df)
        mask = self.df.duplicated(subset=keys, keep=False)
        if verbose and mask.sum() > 0:
            print(f"[remove_near_duplicates] {mask.sum()} rows form near-duplicate pairs on {keys}:")

        self.df = (
            self.df
            .sort_values("Id")                           # df_task2 IDs are lower -> kept
            .drop_duplicates(subset=keys, keep="first")
            .reset_index(drop=True)
        )
        removed = before - len(self.df)
        self.dedup_log.append({
            "step"        : f"near-duplicates ({' + '.join(keys)})",
            "rows_before" : before,
            "rows_removed": removed,
            "rows_after"  : len(self.df),
            "detail"      : "sorted by Id asc; kept df_task2 record when conflict",
        })
        if verbose:
            print(f"[remove_near_duplicates] removed {removed} rows -> {len(self.df)} remaining")

    def validate_global_key(self, key_col: str = "Id", verbose: bool = False) -> bool:
        """Check that key_col is unique across the entire unified table.

        Args:
            key_col: Column to validate as a global unique key.
            verbose: If True, print a detailed uniqueness report.

        Returns:
            True if key_col is unique in self.df, else False.
        """
        n_total  = len(self.df)
        n_unique = self.df[key_col].nunique()
        is_valid = n_total == n_unique
        if verbose:
            print(f"[validate_global_key] column    : '{key_col}'")
            print(f"[validate_global_key] total rows: {n_total}")
            print(f"[validate_global_key] unique    : {n_unique}")
            print(f"[validate_global_key] is unique : {is_valid}")
            if not is_valid:
                dups = self.df[self.df.duplicated(subset=[key_col], keep=False)]
                print("[validate_global_key] duplicate sample:")
                print(dups.head())
        return is_valid

## Task 3. Data Integration

### 3.1 Examining and loading data

Examine `s4196976_data_4_solution.csv` (df_task2) and `data_4.csv` (df_task3) before integrating.

In [5]:
# Load df_task2 - cleaned output from Tasks 1 & 2
df_task2 = pd.read_csv(
    "output/s4196976_data_4_solution.csv",
    parse_dates=["OpenDate", "CloseDate"],
)

# Load df_task3 - raw second dataset from www.jobhuntlisting.com
df_task3 = pd.read_csv("data/data_4.csv")

print(f"df_task2 shape: {df_task2.shape}")
print(f"df_task3 shape: {df_task3.shape}")

df_task2 shape: (50750, 11)
df_task3 shape: (5000, 9)


In [6]:
df_task2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50750 entries, 0 to 50749
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Id            50750 non-null  int64         
 1   Title         50750 non-null  object        
 2   Location      50750 non-null  object        
 3   Company       50750 non-null  object        
 4   ContractType  50750 non-null  object        
 5   ContractTime  50750 non-null  object        
 6   Category      50750 non-null  object        
 7   Salary        50750 non-null  float64       
 8   OpenDate      50750 non-null  datetime64[ns]
 9   CloseDate     50750 non-null  datetime64[ns]
 10  SourceName    50750 non-null  object        
dtypes: datetime64[ns](2), float64(1), int64(1), object(7)
memory usage: 4.3+ MB


In [7]:
df_task2.head(3)

,Id,Title,Location,Company,ContractType,ContractTime,Category,Salary,OpenDate,CloseDate,SourceName
0,69793814,PR Senior Account Executive Consumer PR,London,Fresh Connect,full_time,permanent,"PR, Advertising & Marketing Jobs",28500.0,2012-06-17 12:00:00,2012-08-16 12:00:00,gorkanajobs.co.uk
1,66796338,Press and Public Affairs Manager,London,Prospect Resourcing,full_time,permanent,"PR, Advertising & Marketing Jobs",45000.0,2013-02-19 00:00:00,2013-03-21 00:00:00,gorkanajobs.co.uk
2,69140453,Senior Account Executive Business / Enterprise,South East London,Unicorn Jobs,full_time,permanent,"PR, Advertising & Marketing Jobs",22000.0,2012-05-26 15:00:00,2012-07-25 15:00:00,gorkanajobs.co.uk


In [8]:
df_task3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Location                    5000 non-null   object 
 1   Job Title                   5000 non-null   object 
 2   Monthly Payment             5000 non-null   float64
 3   Closing                     5000 non-null   object 
 4   Category                    5000 non-null   object 
 5   Type                        3593 non-null   object 
 6   Opening                     5000 non-null   object 
 7   Organisation                4531 non-null   object 
 8   Full-Time Equivalent (FTE)  5000 non-null   float64
dtypes: float64(2), object(7)
memory usage: 351.7+ KB


In [9]:
df_task3.head(3)

,Location,Job Title,Monthly Payment,Closing,Category,Type,Opening,Organisation,Full-Time Equivalent (FTE)
0,Berkshire,Lead CRA UK,4583.33,2012-03-08 12:00:00,Health,NaN,2012-01-08 12:00:00,SEC Recruitment,1.0
1,Bristol,Possession Manager,2812.50,2013-09-06 12:00:00,Engineering,Permanent,2013-08-07 12:00:00,Navartis Limited,1.0
2,Coventry,NVQ Assessor Banking/Financial Services Salary...,1791.67,2013-05-02 00:00:00,Hospitality,Permanent,2013-02-01 00:00:00,Pertemps,1.0


### 3.2 Resolving schema conflicts

Schema comparison between the two datasets:

| Global schema (df_task2) | df_task3 column              | Conflict / note |
|--------------------------|------------------------------|-----------------|
| `Id`                     | *(absent)*                   | Must be generated |
| `Title`                  | `Job Title`                  | Naming conflict |
| `Location`               | `Location`                   | No conflict |
| `Company`                | `Organisation`               | Naming conflict; has NaN |
| `ContractType`           | `Full-Time Equivalent (FTE)` | Naming + type: FTE derived |
| `ContractTime`           | `Type`                       | Naming + value conflict |
| `Category`               | `Category`                   | Value conflict (abbreviated) |
| `Salary`                 | `Monthly Payment`            | Naming + unit conflict |
| `OpenDate`               | `Opening`                    | Naming conflict |
| `CloseDate`              | `Closing`                    | Naming conflict |
| `SourceName`             | *(absent)*                   | Must be added |

#### Conflict 1 - Column naming

Every column in df_task3 either has a different name from the global schema, or is absent entirely.

In [10]:
print("Global schema columns:", list(df_task2.columns))
print("df_task3 columns     :", list(df_task3.columns))

Global schema columns: ['Id', 'Title', 'Location', 'Company', 'ContractType', 'ContractTime', 'Category', 'Salary', 'OpenDate', 'CloseDate', 'SourceName']
df_task3 columns     : ['Location', 'Job Title', 'Monthly Payment', 'Closing', 'Category', 'Type', 'Opening', 'Organisation', 'Full-Time Equivalent (FTE)']


#### Conflict 2 - Category value mismatch

df_task3 uses abbreviated sector labels; the global schema requires full canonical strings.

In [11]:
print("df_task3 Category values:", sorted(df_task3["Category"].unique()))
print("df_task2 Category values:", sorted(df_task2["Category"].unique()))

df_task3 Category values: ['Education', 'Engineering', 'Finance', 'Health', 'Hospitality', 'Information Technology', 'Marketing', 'Sales']
df_task2 Category values: ['Accounting & Finance Jobs', 'Engineering Jobs', 'Healthcare & Nursing Jobs', 'Hospitality & Catering Jobs', 'IT Jobs', 'PR, Advertising & Marketing Jobs', 'Sales Jobs', 'Teaching Jobs']


#### Conflict 3 - Salary unit (monthly vs annual)

df_task3 stores `Monthly Payment`; the global schema stores annual `Salary`. Each monthly value is multiplied by 12.

In [12]:
print("df_task3 Monthly Payment - describe:")
print(df_task3["Monthly Payment"].describe())
print("\nEquivalent annual range: [{:.2f}, {:.2f}]".format(
    df_task3["Monthly Payment"].min() * 12,
    df_task3["Monthly Payment"].max() * 12,
))
print("\ndf_task2 Salary - describe:")
print(df_task2["Salary"].describe())

df_task3 Monthly Payment - describe:
count    5000.000000
mean     2854.147972
std      1310.434763
min       424.000000
25%      1916.670000
50%      2583.330000
75%      3541.670000
max      8000.000000
Name: Monthly Payment, dtype: float64

Equivalent annual range: [5088.00, 96000.00]

df_task2 Salary - describe:
count    50750.000000
mean     33756.683970
std      15189.430456
min          0.000000
25%      23000.000000
50%      31000.000000
75%      42500.000000
max      71750.000000
Name: Salary, dtype: float64


#### Conflict 4 - ContractType derived from FTE

df_task3 has no `ContractType` column. `Full-Time Equivalent (FTE)` encodes the same information: 1.0 = full-time, < 1.0 = part-time.

In [13]:
print("df_task3 FTE unique values :", sorted(df_task3["Full-Time Equivalent (FTE)"].dropna().unique()))
print("df_task2 ContractType unique:", df_task2["ContractType"].unique())

df_task3 FTE unique values : [np.float64(0.2), np.float64(0.4), np.float64(0.6), np.float64(0.8), np.float64(1.0)]
df_task2 ContractType unique: ['full_time' 'non-specified' 'part_time']


#### Conflict 5 - ContractTime value normalisation

df_task3 `Type` contains `'Permanent'`, `'Fixed Term Contract'`, and `NaN`; these must map to `permanent`, `contract`, `non-specified`.

In [14]:
print("df_task3 Type unique values :", df_task3["Type"].value_counts(dropna=False).to_dict())
print("df_task2 ContractTime unique:", df_task2["ContractTime"].unique())

df_task3 Type unique values : {'Permanent': 3034, nan: 1407, 'Fixed Term Contract': 559}
df_task2 ContractTime unique: ['permanent' 'contract' 'non-specified']


#### Conflict 6 - Missing Id

df_task3 has no `Id` column. New 8-digit IDs will be generated starting from `max(df_task2['Id']) + 1`.

In [15]:
print(f"df_task2 Id range: [{df_task2['Id'].min()}, {df_task2['Id'].max()}]")
print(f"Next available Id: {df_task2['Id'].max() + 1}")

df_task2 Id range: [12612628, 72705203]
Next available Id: 72705204


### 3.3 Applying schema resolution via SchemaResolver

In [16]:
schema_resolver = SchemaResolver(df_task2, df_task3)
schema_resolver.resolve_all(existing_ids=df_task2["Id"], verbose=True)

[resolve_column_names] Applied: {'Job Title': 'Title', 'Monthly Payment': 'Salary', 'Opening': 'OpenDate', 'Closing': 'CloseDate', 'Organisation': 'Company', 'Type': 'ContractTime', 'Full-Time Equivalent (FTE)': 'ContractType'}
[resolve_categories] df_task3 categories (unchanged): ['Health', 'Engineering', 'Hospitality', 'Information Technology', 'Finance', 'Marketing', 'Sales', 'Education']
[resolve_categories] df_task2 before: ['PR, Advertising & Marketing Jobs', 'Teaching Jobs', 'Engineering Jobs', 'Sales Jobs', 'Accounting & Finance Jobs', 'Hospitality & Catering Jobs', 'IT Jobs', 'Healthcare & Nursing Jobs']
[resolve_categories] df_task2 after : ['Marketing', 'Education', 'Engineering', 'Sales', 'Finance', 'Hospitality', 'Information Technology', 'Health']
[resolve_salary] monthly range: [424.00, 8000.00]
[resolve_salary] annual  range: [5088.00, 96000.00]
[resolve_contract_type] distribution: {'full_time': 4857, 'part_time': 143}
[resolve_contract_time] before: {'Permanent': 3034

In [17]:
schema_resolver.df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Id            5000 non-null   int64         
 1   Title         5000 non-null   object        
 2   Location      5000 non-null   object        
 3   Company       5000 non-null   object        
 4   ContractType  5000 non-null   object        
 5   ContractTime  5000 non-null   object        
 6   Category      5000 non-null   object        
 7   Salary        5000 non-null   float64       
 8   OpenDate      5000 non-null   datetime64[ns]
 9   CloseDate     5000 non-null   datetime64[ns]
 10  SourceName    5000 non-null   object        
dtypes: datetime64[ns](2), float64(1), int64(1), object(7)
memory usage: 429.8+ KB


In [18]:
schema_resolver.df.head(3)

,Id,Title,Location,Company,ContractType,ContractTime,Category,Salary,OpenDate,CloseDate,SourceName
0,72705204,Lead CRA UK,Berkshire,SEC Recruitment,full_time,non-specified,Health,54999.96,2012-01-08 12:00:00,2012-03-08 12:00:00,www.jobhuntlisting.com
1,72705205,Possession Manager,Bristol,Navartis Limited,full_time,permanent,Engineering,33750.00,2013-08-07 12:00:00,2013-09-06 12:00:00,www.jobhuntlisting.com
2,72705206,NVQ Assessor Banking/Financial Services Salary...,Coventry,Pertemps,full_time,permanent,Hospitality,21500.04,2013-02-01 00:00:00,2013-05-02 00:00:00,www.jobhuntlisting.com


In [19]:
# Schema resolution log - one row per conflict resolved
schema_resolver.get_schema_log_df()

,conflict,action,detail
0,naming,renamed columns,"{'Job Title': 'Title', 'Monthly Payment': 'Sal..."
1,category values,converted df_task2 canonical labels to general...,"df_task2 before=['PR, Advertising & Marketing ..."
2,salary unit,"converted monthly -> annual (x12, rounded 2 dp)","range [424.0, 8000.0] -> [5088.0, 96000.0]"
3,ContractType (FTE derivation),"FTE 1.0->full_time, <1.0->part_time, NaN->non-...","{'full_time': 4857, 'part_time': 143}"
4,ContractTime values,"normalised to {permanent, contract, non-specif...","before={'Permanent': 3034, nan: 1407, 'Fixed T..."
5,missing Id,generated sequential IDs from 72705204 to 7271...,collision check: 0 overlaps
6,missing SourceName,set SourceName = 'www.jobhuntlisting.com' for ...,
7,missing Company,filled 469 NaN values with 'non-specified',
8,date dtype,parsed OpenDate and CloseDate to datetime64,
9,column order,reordered to global schema,"['Id', 'Title', 'Location', 'Company', 'Contra..."


### 3.4 Merging the two datasets

In [20]:
integrator = DataIntegrator(schema_resolver.df_task2, schema_resolver.df)
print(f"Unified shape: {integrator.df.shape}  (expected {len(df_task2)} + {len(df_task3)} = {len(df_task2)+len(df_task3)})")
integrator.df.info()

Unified shape: (55750, 11)  (expected 50750 + 5000 = 55750)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55750 entries, 0 to 55749
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Id            55750 non-null  int64         
 1   Title         55750 non-null  object        
 2   Location      55750 non-null  object        
 3   Company       55750 non-null  object        
 4   ContractType  55750 non-null  object        
 5   ContractTime  55750 non-null  object        
 6   Category      55750 non-null  object        
 7   Salary        55750 non-null  float64       
 8   OpenDate      55750 non-null  datetime64[ns]
 9   CloseDate     55750 non-null  datetime64[ns]
 10  SourceName    55750 non-null  object        
dtypes: datetime64[ns](2), float64(1), int64(1), object(7)
memory usage: 4.7+ MB


## 4. Resolving data-level conflicts

### 4.1 Complete duplicates

A *complete duplicate* is a row where every column value is identical to another row, representing the same job ad ingested twice.

In [21]:
n_complete_dups = integrator.df.duplicated(keep=False).sum()
print(f"Complete duplicate rows (all columns identical): {n_complete_dups}")

Complete duplicate rows (all columns identical): 0


In [22]:
integrator.remove_complete_duplicates(verbose=True)

[remove_complete_duplicates] removed 0 rows -> 55750 remaining


### 4.2 Near-duplicates

A *near-duplicate* is a pair of rows describing the same job ad from different sources with a different `Id`. We detect them via composite key: `Title + Company + Location + OpenDate`.

In [23]:
near_dup_keys = ["Title", "Company", "Location", "OpenDate"]
n_near_dups = integrator.df.duplicated(subset=near_dup_keys, keep=False).sum()
print(f"Near-duplicate rows (same {near_dup_keys}): {n_near_dups}")

Near-duplicate rows (same ['Title', 'Company', 'Location', 'OpenDate']): 594


In [24]:
# Inspect a sample of near-duplicate pairs before removal
near_dup_mask = integrator.df.duplicated(subset=near_dup_keys, keep=False)
integrator.df[near_dup_mask].sort_values(near_dup_keys).head(6)

,Id,Title,Location,Company,ContractType,ContractTime,Category,Salary,OpenDate,CloseDate,SourceName
3483,68704989,1st/2nd Line Support Analyst,Blackpool,Modis,non-specified,contract,Information Technology,30720.00,2013-08-24 00:00:00,2013-09-07 00:00:00,totaljobs.com
54085,72708539,1st/2nd Line Support Analyst,Blackpool,Modis,full_time,contract,Information Technology,30720.00,2013-08-24 00:00:00,2013-09-07 00:00:00,www.jobhuntlisting.com
11718,72690312,"2nd Line Support Engineer Citrix, VMware Ess...",London,Amrec Recruitment Ltd,non-specified,permanent,Information Technology,35000.00,2012-10-16 12:00:00,2013-01-14 12:00:00,jobsite.co.uk
53535,72707989,"2nd Line Support Engineer Citrix, VMware Ess...",London,Amrec Recruitment Ltd,full_time,permanent,Information Technology,35000.04,2012-10-16 12:00:00,2013-01-14 12:00:00,www.jobhuntlisting.com
6425,68062545,A New Professional Career Part Time / Full Time,Crawley,Hiredonline,part_time,non-specified,Sales,15000.00,2013-06-23 00:00:00,2013-08-22 00:00:00,totaljobs.com
53316,72707770,A New Professional Career Part Time / Full Time,Crawley,Hiredonline,part_time,non-specified,Sales,15000.00,2013-06-23 00:00:00,2013-08-22 00:00:00,www.jobhuntlisting.com


In [25]:
integrator.remove_near_duplicates(keys=near_dup_keys, verbose=True)

[remove_near_duplicates] 594 rows form near-duplicate pairs on ['Title', 'Company', 'Location', 'OpenDate']:
[remove_near_duplicates] removed 297 rows -> 55453 remaining


In [26]:
# Deduplication summary log
integrator.get_dedup_log_df()

,step,rows_before,rows_removed,rows_after,detail
0,complete duplicates,55750,0,55750,all-column exact match; kept first occurrence
1,near-duplicates (Title + Company + Location + ...,55750,297,55453,sorted by Id asc; kept df_task2 record when co...


### 4.3 Finding the global key

**Chosen key: `Id`**

**Justification:**
- All df_task2 records carried an `Id` assigned by source job boards (8-digit integers, verified unique in Task 2).
- New `Id` values for df_task3 were generated sequentially from `max(df_task2.Id) + 1`, guaranteeing no overlap.
- After both deduplication steps, no two rows share the same `Id`.
- `Id` is a single-column surrogate key — efficient for joins and lookups, immune to text variants (typos, case, truncation) that would affect a composite natural key like `(Title, Company, Location, OpenDate)`.

In [27]:
integrator.validate_global_key(key_col="Id", verbose=True)

[validate_global_key] column    : 'Id'
[validate_global_key] total rows: 55453
[validate_global_key] unique    : 55453
[validate_global_key] is unique : True


True

In [28]:
integrator.df.describe(include="all")

,Id,Title,Location,Company,ContractType,ContractTime,Category,Salary,OpenDate,CloseDate,SourceName
count,5.545300e+04,55453,55453,55453,55453,55453,55453,55453.000000,55453,55453,55453
unique,NaN,55450,496,9093,3,3,8,NaN,NaN,NaN,108
top,NaN,Professional Liabilities Claims Handler,UK,non-specified,non-specified,permanent,Information Technology,NaN,NaN,NaN,totaljobs.com
freq,NaN,2,8419,5388,38068,33872,14354,NaN,NaN,NaN,9224
mean,6.917424e+07,NaN,NaN,NaN,NaN,NaN,NaN,33801.108532,2012-12-31 02:31:24.558094336,2013-02-22 00:42:31.544551168,NaN
min,1.261263e+07,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,2012-01-01 00:00:00,2012-01-15 12:00:00,NaN
25%,6.845125e+07,NaN,NaN,NaN,NaN,NaN,NaN,23000.000000,2012-07-01 00:00:00,2012-08-22 12:00:00,NaN
50%,6.959684e+07,NaN,NaN,NaN,NaN,NaN,NaN,31000.000000,2012-12-31 12:00:00,2013-02-21 15:00:00,NaN
75%,7.164946e+07,NaN,NaN,NaN,NaN,NaN,NaN,42500.000000,2013-07-02 00:00:00,2013-08-24 15:00:00,NaN
max,7.271020e+07,NaN,NaN,NaN,NaN,NaN,NaN,96000.000000,2014-01-17 15:00:00,2014-03-31 15:00:00,NaN


In [29]:
integrator.df.head(5)

,Id,Title,Location,Company,ContractType,ContractTime,Category,Salary,OpenDate,CloseDate,SourceName
0,12612628,Engineering Systems Analyst,Dorking,Gregory Martin International,non-specified,permanent,Engineering,25000.0,2012-11-03 00:00:00,2012-12-03 00:00:00,cv-library.co.uk
1,12612830,Stress Engineer Glasgow,Glasgow,Gregory Martin International,non-specified,permanent,Engineering,30000.0,2013-01-08 15:00:00,2013-04-08 15:00:00,cv-library.co.uk
2,12612844,Modelling and simulation analyst,Hampshire,Gregory Martin International,non-specified,permanent,Engineering,30000.0,2013-07-26 15:00:00,2013-09-24 15:00:00,cv-library.co.uk
3,12613049,Engineering Systems Analyst / Mathematical Mod...,Surrey,Gregory Martin International,non-specified,permanent,Engineering,27500.0,2012-12-14 00:00:00,2013-03-14 00:00:00,cv-library.co.uk
4,12613647,"Pioneer, Miser Engineering Systems Analyst",Surrey,Gregory Martin International,non-specified,permanent,Engineering,25000.0,2013-10-25 00:00:00,2013-12-24 00:00:00,cv-library.co.uk


## 5. Saving the integrated dataset

In [30]:
student_id  = "s4196976"
output_path = f"output/{student_id}_data_4_integrated.csv"

integrator.df.to_csv(output_path, index=False)
print(f"Saved {len(integrator.df):,} rows -> {output_path}")

Saved 55,453 rows -> output/s4196976_data_4_integrated.csv


## Summary

This task integrated two heterogeneous job advertisement datasets using a class-based pipeline:

- **`SemanticLayer`** — pure mapping class holding all translation rules (category labels, ContractTime values, column rename map, FTE->ContractType logic, salary unit conversion).
- **`SchemaResolver`** — stateful class owning `self.df` (raw df_task3). Each `resolve_*` method fixes one schema conflict in place and appends to `self.schema_log`. `resolve_all()` chains them in the correct order.
- **`DataIntegrator`** — stateful class owning `self.df` (the merged table). `remove_complete_duplicates()` and `remove_near_duplicates()` mutate it in place while logging to `self.dedup_log`. `validate_global_key()` confirms `Id` uniqueness.

| Step | Action | Result |
|------|--------|--------|
| Load | df_task2 (50 750) + df_task3 (5 000) | - |
| Schema resolution | 7 conflicts resolved via `SchemaResolver.resolve_all` | 5 000 rows aligned |
| Merge | `pd.concat` | 55 750 rows |
| Complete deduplication | exact-match removal | 0 removed |
| Near-deduplication | Title+Company+Location+OpenDate key | 297 removed |
| **Final table** | **`s4196976_data_4_integrated.csv`** | **55 453 rows, `Id` unique** |